In [1]:
from ml_recruitment import ML_Recruitment
import os
from numpy import nan

vCaminhoBase = os.path.join("..", 'etl', "output", "dataset_unificado_balanceado.csv")

vML = ML_Recruitment(pCaminhoBase=vCaminhoBase)

# Carregar a base de dados
vML.carregarBase()

# Criar uma coluna de teste com valores NaN
vML.baseDeDados['TesTE de Nome de ColunA'] = nan

# Padronizar a base de dados
vML.padronizarBase(pListaDeColunasParaFillNA=['requisitos_vaga', 'cv_texto', 'TesTE de Nome de ColunA'], pPreencherNulosCom='vazio', pPadronizarColunasTexto=False)

# Treinar o vetorizador textual
vML.treinarVetorizadorTextual(pListaColunas=['requisitos_vaga', 'cv_texto'], pNumeroMaximoFeatures=300)

# Calcular a similaridade textual
vML.baseDeDados['sim_textual'] = vML.calcularSimilaridadeTextual(pListaColunas=['requisitos_vaga', 'cv_texto'])

# Inserir novas colunas
vML.baseDeDados['match_nivel'] = (vML.baseDeDados['nivel_profissional_vaga'] == vML.baseDeDados['nivel_profissional_candidato']).astype(int)
vML.baseDeDados['match_ingles'] = (vML.baseDeDados['nivel_ingles_vaga'] == vML.baseDeDados['nivel_ingles_candidato']).astype(int)
vML.baseDeDados['match_profissional'] = (vML.baseDeDados['nivel_profissional_vaga'] == vML.baseDeDados['nivel_profissional_candidato']).astype(int)
vML.baseDeDados['match_espanhol'] = (vML.baseDeDados['nivel_espanhol_vaga'] == vML.baseDeDados['nivel_espanhol_candidato']).astype(int)
vML.baseDeDados['match_local'] = (vML.baseDeDados['local_vaga'] == vML.baseDeDados['local_candidato']).astype(int)
vML.baseDeDados['match_academico'] = (vML.baseDeDados['nivel_academico_vaga'] == vML.baseDeDados['nivel_academico_candidato']).astype(int)

# Separar features e target
vML.separarFeatureTarget(pColunaTarget='match', pColunasIgnorar=['situacao', 'comentario', 'recrutador', 'vaga_id', 'codigo_candidato', 'nome_candidato', 'conhecimentos_tecnicos', 'teste_de_nome_de_coluna'])

# Criar a pipeline de pré-processamento
vML.criarPipeline(
    pColunasNumericas=['sim_textual', 'match_nivel', 'match_ingles', 'match_profissional', 'match_espanhol', 'match_local', 'match_academico'],
    pColunasCategoricas=['titulo_vaga', 'nivel_profissional_vaga', 'nivel_ingles_vaga', 'nivel_espanhol_vaga', 'nivel_academico_vaga', 'nivel_academico_candidato', 'nivel_ingles_candidato', 'nivel_espanhol_candidato', 'nivel_profissional_candidato', 'local_candidato', 'cliente', 'local_vaga'],
    pColunasTexto=['requisitos_vaga', 'cv_texto'],
    pNumeroMaximoFeatures=30
)

# Separar a base de dados em treino e teste
vML.separarTreinoTeste(pProporcaoTreino=0.2, pRandomState=42)

# Executar a pipeline de pré-processamento
vML.executarPipeline(pAplicarEm='Treino_Teste')

# Criar o modelo XGBoost
vTotalPos = vML.target_Treino.sum()
vScalePosWeight = (len(vML.target_Treino) - vTotalPos) / vTotalPos if vTotalPos > 0 else 1

vParametrosXGB = {
    'n_estimators': 100,
    'eval_metric': 'logloss',
    'scale_pos_weight': vScalePosWeight,
    'use_label_encoder': False,
    'early_stopping_rounds': 10,
    'random_state': 42
}

vML.criarModeloXGB(pParametros=vParametrosXGB, pTreinarModelo=False)

# Treinar o modelo XGBoost
vML.treinarModeloXGB()

# Avaliar o modelo XGBoost
print(vML.avaliarModeloXGB(pOutputDict=False))

# Salvar vetorizador, pipeline e modelo
vML.salvarVetorizadorTextual(pCaminhoArquivo=os.path.join("output", "vetorizador_similaridade_textual.pkl"))
vML.salvarPipeline(pCaminhoArquivo=os.path.join("output", "preprocessador_xgb.pkl"))
vML.salvarModeloXGB(pCaminhoArquivo=os.path.join("output", "modelo_xgb.pkl"))


c:\Users\rafae\OneDrive\01_Documentos\13_Projetos\venv_api\Lib\site-packages\xgboost\callback.py:386: UserWarning: [15:16:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


              precision    recall  f1-score   support

           0      0.695     0.831     0.757       639
           1      0.790     0.635     0.704       639

    accuracy                          0.733      1278
   macro avg      0.742     0.733     0.731      1278
weighted avg      0.742     0.733     0.731      1278



In [3]:
vML.prever(pFeatures=vML.features_Teste)

array([1, 0, 0, ..., 1, 0, 1], shape=(1278,))

In [4]:
print(vML)

Caminho da Base: ..\etl\output\dataset_unificado_balanceado.csv
Número de Linhas: 6386
Número de Colunas: 30
Tamanho das features: (6386, 21)
Tamanho do target: (6386,)
Número de Features de Treino: (5108, 21)
Número de Target de Treino: (5108,)
Número de Features de Teste: (1278, 21)
Número de Target de Teste: (1278,)
Pipeline: ColumnTransformer(sparse_threshold=1.0,
                  transformers=[('numerical',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['sim_textual', 'match_nivel', 'match_ingles',
                                  'match_profissional', 'match_espanhol',
                                  'match_local', 'match_academico']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='missing',
    